# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [21]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, GradientBoostingClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import accuracy_score
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV


In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
spaceship_clean = spaceship.dropna()
spaceship_clean["Cabin"] = spaceship_clean["Cabin"].str.split("/").str[0]
spaceship_clean["Cabin"].head()

C:\Users\Ready2Use\AppData\Local\Temp\ipykernel_19052\2681450784.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  spaceship_clean["Cabin"] = spaceship_clean["Cabin"].str.split("/").str[0]


0    B
1    F
2    A
3    A
4    F
Name: Cabin, dtype: object

In [4]:
spaceship_clean = spaceship_clean.drop(["PassengerId", "Name"], axis= 1)
spaceship_clean.head()

,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported
0,Europa,False,B,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False
1,Earth,False,F,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True
2,Europa,False,A,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False
3,Europa,False,A,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False
4,Earth,False,F,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,True


In [5]:
# 1. Separate Features and Target
X = spaceship_clean.drop('Transported', axis=1)
y = spaceship_clean['Transported']

# 2. Identify Columns
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object', 'bool']).columns

print(f"Numeric: {list(numeric_features)}")
print(f"Categorical: {list(categorical_features)}")

# 3. Define Transformers
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Just in case any NaNs remain
    ('scaler', StandardScaler())                   # FEATURE SCALING
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')) # FEATURE ENCODING
])

# 4. Combine and Apply
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Fit and Transform
X_processed = preprocessor.fit_transform(X)

# Convert to dense array for the model (needed for some sklearn versions)
if hasattr(X_processed, "toarray"):
    X_dense = X_processed.toarray()
else:
    X_dense = X_processed

print(f"Processed Shape: {X_dense.shape}")

Numeric: ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
Categorical: ['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP']
Processed Shape: (6606, 24)


In [6]:
# 1. Initialize a Random Forest (used only to find important features)
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# 2. Configure the Selector
# threshold='mean' keeps only features with importance > average importance
selector = SelectFromModel(rf, threshold='mean', prefit=False)

# 3. Fit and Transform
X_selected = selector.fit_transform(X_dense, y)

print(f"✅ Feature Selection Complete.")
print(f"Features before: {X_dense.shape[1]}")
print(f"Features after:  {X_selected.shape[1]}")
print(f"Removed: {X_dense.shape[1] - X_selected.shape[1]} features")

# Store the final ready-to-use data
X_final = X_selected
y_final = y

✅ Feature Selection Complete.
Features before: 24
Features after:  8
Removed: 16 features


In [7]:
# Split the data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y_final, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

Training set size: 5284
Test set size: 1322


In [8]:
# Base estimator: A single Decision Tree
base_tree = DecisionTreeClassifier(random_state=42)

# 1. Bagging Classifier (bootstrap=True)
# Samples with replacement
bagging_clf = BaggingClassifier(
    estimator=base_tree,
    n_estimators=100,
    max_samples=1.0,      # Use 100% of the data for each tree
    bootstrap=True,       # <--- KEY: Sampling WITH replacement
    n_jobs=-1,
    random_state=42
)

# 2. Pasting Classifier (bootstrap=False)
# Samples without replacement
pasting_clf = BaggingClassifier(
    estimator=base_tree,
    n_estimators=100,
    max_samples=1.0,      # Use 100% of the data for each tree
    bootstrap=False,      # <--- KEY: Sampling WITHOUT replacement
    n_jobs=-1,
    random_state=42
)

# Train both models
print("Training Bagging model...")
bagging_clf.fit(X_train, y_train)

print("Training Pasting model...")
pasting_clf.fit(X_train, y_train)

Training Bagging model...
Training Pasting model...


,estimator,DecisionTreeC...ndom_state=42)
,n_estimators,100
,max_samples,1.0
,max_features,1.0
,bootstrap,False
,bootstrap_features,False
,oob_score,False
,warm_start,False
,n_jobs,-1
,random_state,42
,verbose,0


In [9]:
# Predictions
y_pred_bagging = bagging_clf.predict(X_test)
y_pred_pasting = pasting_clf.predict(X_test)

# Accuracy Scores
acc_bagging = accuracy_score(y_test, y_pred_bagging)
acc_pasting = accuracy_score(y_test, y_pred_pasting)

# Baseline: Single Decision Tree (for reference)
single_tree = DecisionTreeClassifier(random_state=42)
single_tree.fit(X_train, y_train)
acc_single = accuracy_score(y_test, single_tree.predict(X_test))

print("-" * 40)
print(f"{'Model':<20} | {'Accuracy':<10}")
print("-" * 40)
print(f"{'Single Decision Tree':<20} | {acc_single:.4f}")
print(f"{'Bagging (With Rep.)':<20} | {acc_bagging:.4f}")
print(f"{'Pasting (No Rep.)':<20} | {acc_pasting:.4f}")
print("-" * 40)

# Determine winner
if acc_bagging > acc_pasting:
    print("Winner: Bagging")
elif acc_pasting > acc_bagging:
    print("Winner: Pasting")
else:
    print("It's a tie!")

----------------------------------------
Model                | Accuracy  
----------------------------------------
Single Decision Tree | 0.7655
Bagging (With Rep.)  | 0.8041
Pasting (No Rep.)    | 0.7625
----------------------------------------
Winner: Bagging


In [10]:
# 1. Define the Random Forest Model
# n_estimators: Number of trees (same as before)
# max_features: 'sqrt' means it considers sqrt(total_features) at each split (Default for classification)
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_features='sqrt',  # <--- KEY: Random feature selection at each split
    n_jobs=-1,
    random_state=42
)

# 2. Train the model
print("Training Random Forest model...")
rf_clf.fit(X_train, y_train)

# 3. Evaluate
y_pred_rf = rf_clf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)

# 4. Update the Comparison Table
print("-" * 40)
print(f"{'Model':<20} | {'Accuracy':<10}")
print("-" * 40)
print(f"{'Single Decision Tree':<20} | {acc_single:.4f}")
print(f"{'Bagging':<20} | {acc_bagging:.4f}")
print(f"{'Pasting':<20} | {acc_pasting:.4f}")
print(f"{'Random Forest':<20} | {acc_rf:.4f}")
print("-" * 40)

# Determine the new winner
models = {
    "Single Decision Tree": acc_single,
    "Bagging": acc_bagging,
    "Pasting": acc_pasting,
    "Random Forest": acc_rf
}

best_model = max(models, key=models.get)
print(f"Current Leader: {best_model} ({models[best_model]:.4f})")

Training Random Forest model...
----------------------------------------
Model                | Accuracy  
----------------------------------------
Single Decision Tree | 0.7655
Bagging              | 0.8041
Pasting              | 0.7625
Random Forest        | 0.8033
----------------------------------------
Current Leader: Bagging (0.8041)


In [11]:
# 1. Define the Gradient Boosting Model
# learning_rate: Shrinks the contribution of each tree (0.1 is standard)
# n_estimators: Number of sequential trees (often needs more than Bagging/RF)
gb_clf = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,          # Limit depth to prevent overfitting (shallow trees)
    random_state=42
)

# 2. Train the model
# Note: This might take slightly longer than Random Forest because it is sequential
print("Training Gradient Boosting model...")
gb_clf.fit(X_train, y_train)

# 3. Evaluate
y_pred_gb = gb_clf.predict(X_test)
acc_gb = accuracy_score(y_test, y_pred_gb)

# 4. Update the Comparison Table
print("-" * 40)
print(f"{'Model':<20} | {'Accuracy':<10}")
print("-" * 40)
print(f"{'Single Decision Tree':<20} | {acc_single:.4f}")
print(f"{'Bagging':<20} | {acc_bagging:.4f}")
print(f"{'Pasting':<20} | {acc_pasting:.4f}")
print(f"{'Random Forest':<20} | {acc_rf:.4f}")
print(f"{'Gradient Boosting':<20} | {acc_gb:.4f}")
print("-" * 40)

# Determine the new winner
models = {
    "Single Decision Tree": acc_single,
    "Bagging": acc_bagging,
    "Pasting": acc_pasting,
    "Random Forest": acc_rf,
    "Gradient Boosting": acc_gb
}

best_model = max(models, key=models.get)

Training Gradient Boosting model...
----------------------------------------
Model                | Accuracy  
----------------------------------------
Single Decision Tree | 0.7655
Bagging              | 0.8041
Pasting              | 0.7625
Random Forest        | 0.8033
Gradient Boosting    | 0.7995
----------------------------------------


In [12]:
# 1. Define the AdaBoost Model
# In sklearn, AdaBoost typically uses a DecisionTree as the base estimator.
# n_estimators: Number of sequential trees
# learning_rate: Controls how much each tree contributes (1.0 is default)
ada_clf = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1), # Often uses "stumps" (depth=1)
    n_estimators=100,
    learning_rate=1.0,
    random_state=42
)

# 2. Train the model
print("Training AdaBoost model...")
ada_clf.fit(X_train, y_train)

# 3. Evaluate
y_pred_ada = ada_clf.predict(X_test)
acc_ada = accuracy_score(y_test, y_pred_ada)

# 4. Final Comparison Table
print("-" * 40)
print(f"{'Model':<20} | {'Accuracy':<10}")
print("-" * 40)
print(f"{'Single Decision Tree':<20} | {acc_single:.4f}")
print(f"{'Bagging':<20} | {acc_bagging:.4f}")
print(f"{'Pasting':<20} | {acc_pasting:.4f}")
print(f"{'Random Forest':<20} | {acc_rf:.4f}")
print(f"{'Gradient Boosting':<20} | {acc_gb:.4f}")
print(f"{'AdaBoost':<20} | {acc_ada:.4f}")
print("-" * 40)

# Determine the Final Winner
models = {
    "Single Decision Tree": acc_single,
    "Bagging": acc_bagging,
    "Pasting": acc_pasting,
    "Random Forest": acc_rf,
    "Gradient Boosting": acc_gb,
    "AdaBoost": acc_ada
}

best_model = max(models, key=models.get)
print(f"FINAL WINNER: {best_model} ({models[best_model]:.4f})")

# Optional: Print a brief summary
if best_model.startswith("Random") or best_model.startswith("Bagging") or best_model.startswith("Pasting"):
    print("\nInsight: Parallel Ensembles (Bagging/RF) performed best. This suggests reducing variance was key.")
elif best_model.startswith("Gradient") or best_model.startswith("Ada"):
    print("\nInsight: Sequential Ensembles (Boosting) performed best. This suggests correcting errors sequentially was key.")

Training AdaBoost model...
----------------------------------------
Model                | Accuracy  
----------------------------------------
Single Decision Tree | 0.7655
Bagging              | 0.8041
Pasting              | 0.7625
Random Forest        | 0.8033
Gradient Boosting    | 0.7995
AdaBoost             | 0.7761
----------------------------------------
FINAL WINNER: Bagging (0.8041)

Insight: Parallel Ensembles (Bagging/RF) performed best. This suggests reducing variance was key.


- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

Step 1: Define the Parameter Grid
We need to decide which settings to tune for the BaggingClassifier. The most impactful ones are:

n_estimators: How many trees to build. (More is usually better, but slower).
max_samples: What fraction of the data each tree sees. (1.0 = 100%, 0.8 = 80%).
max_features: What fraction of features each tree sees. (1.0 = all features, 0.8 = 80%).

In [ ]:
# 1. Define the Base Model
# We use the same Decision Tree as before
base_tree = DecisionTreeClassifier(random_state=42)

# 2. Define the Parameter Grid
# This creates a list of every combination to test.
# Total combinations = 2 (n_estimators) * 2 (max_samples) * 2 (max_features) = 8 models
param_grid = {
    'n_estimators': [50, 100],          # Try 50 vs 100 trees
    'max_samples': [0.8, 1.0],          # Try 80% vs 100% of data samples
    'max_features': [0.8, 1.0]          # Try 80% vs 100% of features
}

# 3. Initialize GridSearchCV
# cv=5 means 5-Fold Cross-Validation (trains 5 times per combination for robustness)
# n_jobs=-1 uses all CPU cores to speed up the process
grid_search = GridSearchCV(
    estimator=BaggingClassifier(estimator=base_tree, random_state=42, n_jobs=-1),
    param_grid=param_grid,
    cv=5,               # 5-Fold Cross-Validation
    scoring='accuracy', # Metric to optimize
    n_jobs=-1,          # Parallel processing
    verbose=1           # Shows progress bar
)

# 4. Run the Search
# This will train 8 combinations * 5 folds = 40 models.
print("Starting Hyperparameter Tuning (this may take a minute)...")
grid_search.fit(X_train, y_train)

# 5. Display Results
print("\n✅ Tuning Complete!")
print(f"🏆 Best Accuracy: {grid_search.best_score_:.4f}")
print(f"⚙️  Best Parameters: {grid_search.best_params_}")

Starting Hyperparameter Tuning (this may take a minute)...
Fitting 5 folds for each of 8 candidates, totalling 40 fits

✅ Tuning Complete!
🏆 Best Accuracy: 0.7884
⚙️  Best Parameters: {'max_features': 0.8, 'max_samples': 0.8, 'n_estimators': 50}


- Evaluate your model

In [ ]:
# 1. Get the Best Model from the Grid Search
best_bagging_model = grid_search.best_estimator_

# 2. Make Predictions on the Test Set
y_pred_tuned = best_bagging_model.predict(X_test)

# 3. Calculate Final Accuracy
final_accuracy = accuracy_score(y_test, y_pred_tuned)

# 4. Compare Results
original_accuracy = 0.8041  # Your previous best Bagging score

print("-" * 50)
print("📊 FINAL EVALUATION RESULTS")
print("-" * 50)
print(f"Best Parameters Found: {grid_search.best_params_}")
print(f"Cross-Val Score (during tuning): {grid_search.best_score_:.4f}")
print(f"Final Test Accuracy (Tuned Model): {final_accuracy:.4f}")
print(f"Original Bagging Accuracy (Untuned): {original_accuracy:.4f}")
print("-" * 50)

# Calculate Improvement
difference = final_accuracy - original_accuracy
if difference > 0:
    print(f"🚀 Improvement: +{difference:.4f} (Tuning Helped!)")
elif difference < 0:
    print(f"📉 Decrease: {difference:.4f} (Tuning didn't help on this specific test split)")
else:
    print("⚖️ No change.")

# Optional: Detailed Report (Precision, Recall, F1)
print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred_tuned))

--------------------------------------------------
📊 FINAL EVALUATION RESULTS
--------------------------------------------------
Best Parameters Found: {'max_features': 0.8, 'max_samples': 0.8, 'n_estimators': 50}
Cross-Val Score (during tuning): 0.7884
Final Test Accuracy (Tuned Model): 0.7965
Original Bagging Accuracy (Untuned): 0.8041
--------------------------------------------------
📉 Decrease: -0.0076 (Tuning didn't help on this specific test split)

📋 Classification Report:
              precision    recall  f1-score   support

       False       0.82      0.75      0.79       653
        True       0.78      0.84      0.81       669

    accuracy                           0.80      1322
   macro avg       0.80      0.80      0.80      1322
weighted avg       0.80      0.80      0.80      1322



**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

Step 1: Run a "Coarse" Grid Search (Fast)
This tests 12 combinations 

In [ ]:

# 1. Define a Smaller, Strategic Grid (Coarse Search)
param_grid_coarse = {
    'n_estimators': [50, 100, 200],       # How many trees?
    'max_samples': [0.5, 0.8, 1.0],       # How much data per tree?
    'max_features': [1.0]                 # Keep features constant for now to save time
    # We leave estimator__max_depth as default (None) for this round
}

# 2. Setup GridSearch
# cv=5 is fine now because the grid is small (12 combos * 5 folds = 60 models)
grid_coarse = GridSearchCV(
    estimator=BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1),
    param_grid=param_grid_coarse,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

# 3. Run the Search
print("🚀 Running Coarse Grid Search (12 combinations)...")
grid_coarse.fit(X_train, y_train)

# 4. Show Results
print("\n✅ Coarse Search Complete!")
print(f"🏆 Best CV Accuracy: {grid_coarse.best_score_:.4f}")
print(f"⚙️  Best Parameters: {grid_coarse.best_params_}")

🚀 Running Coarse Grid Search (12 combinations)...
Fitting 5 folds for each of 9 candidates, totalling 45 fits

✅ Coarse Search Complete!
🏆 Best CV Accuracy: 0.7884
⚙️  Best Parameters: {'max_features': 1.0, 'max_samples': 0.5, 'n_estimators': 50}


- Run Grid Search

Step 2: The "Fine" Grid Search
Now we zoom in. We know max_samples=0.5 is good. Let's keep that fixed (or test very close values) and now introduce max_depth. Limiting the depth of individual trees is often the missing piece to prevent overfitting within the ensemble.

We will test depths like 5, 10, 15, and None (unlimited), while keeping the other winning parameters in mind.

In [23]:
# 1. Define the "Fine" Grid
# We focus on tree complexity (max_depth) and verify the sample size
param_grid_fine = {
    'n_estimators': [50, 100],            # Verify if 100 helps slightly with depth control
    'max_samples': [0.5, 0.6],            # Zoom in around the winner (0.5)
    'max_features': [1.0],                # Keep constant based on previous results
    'estimator__max_depth': [5, 10, 15, None] # The new critical parameter!
}

# 2. Setup GridSearch
# Total combos: 2 * 2 * 1 * 4 = 16 candidates. 
# 16 * 5 folds = 80 fits. Very manageable.
grid_fine = GridSearchCV(
    estimator=BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1),
    param_grid=param_grid_fine,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

# 3. Run the Fine Search
print("🔍 Running Fine Grid Search (focusing on tree depth)...")
grid_fine.fit(X_train, y_train)

# 4. Show Final Results
print("\n✅ Fine Search Complete!")
print(f"🏆 Best CV Accuracy: {grid_fine.best_score_:.4f}")
print(f"⚙️  Best Parameters: {grid_fine.best_params_}")

# Compare with previous best
prev_best = 0.7884
if grid_fine.best_score_ > prev_best:
    print(f"🚀 Improvement! +{grid_fine.best_score_ - prev_best:.4f}")
else:
    print("ℹ️  Score is stable. The previous parameters were already near optimal.")

🔍 Running Fine Grid Search (focusing on tree depth)...
Fitting 5 folds for each of 16 candidates, totalling 80 fits

✅ Fine Search Complete!
🏆 Best CV Accuracy: 0.7947
⚙️  Best Parameters: {'estimator__max_depth': 5, 'max_features': 1.0, 'max_samples': 0.6, 'n_estimators': 50}
🚀 Improvement! +0.0063


- Evaluate your model

In [ ]:
# 1. Extract the Best Model
final_best_model = grid_fine.best_estimator_

# 2. Predict on the Test Set
y_pred_final = final_best_model.predict(X_test)

# 3. Calculate Final Accuracy
final_test_accuracy = accuracy_score(y_test, y_pred_final)

# 4. Comparison
original_untuned_score = 0.8041
previous_cv_score = 0.7949

print("=" * 50)
print("🏆 FINAL HYPER-TUNING EVALUATION")
print("=" * 50)
print(f"Best Parameters: {grid_fine.best_params_}")
print(f"Cross-Val Score (Tuned): {previous_cv_score:.4f}")
print(f"Final Test Accuracy (Tuned): {final_test_accuracy:.4f}")
print(f"Original Test Accuracy (Untuned): {original_untuned_score:.4f}")
print("=" * 50)

diff = final_test_accuracy - original_untuned_score
if diff > 0:
    print(f"🚀 SUCCESS! Tuning improved test accuracy by +{diff:.4f}")
elif diff < 0:
    print(f"⚠️  Note: Test accuracy dropped slightly by {diff:.4f}.")
    print("   (This can happen due to random split variance, but the CV score is more reliable).")
else:
    print("⚖️  Identical performance.")

print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred_final))

🏆 FINAL HYPER-TUNING EVALUATION
Best Parameters: {'estimator__max_depth': 5, 'max_features': 1.0, 'max_samples': 0.5, 'n_estimators': 100}
Cross-Val Score (Tuned): 0.7949
Final Test Accuracy (Tuned): 0.8079
Original Test Accuracy (Untuned): 0.8041
🚀 SUCCESS! Tuning improved test accuracy by +0.0038

📋 Classification Report:
              precision    recall  f1-score   support

       False       0.83      0.77      0.80       653
        True       0.79      0.84      0.82       669

    accuracy                           0.81      1322
   macro avg       0.81      0.81      0.81      1322
weighted avg       0.81      0.81      0.81      1322

